# 01 — Exploratory Data Analysis
## Diabetes Risk Stratification · CDC BRFSS 2015

**Author:** Alejandro Zakzuk — Physician · AI Applied to Health  
**Dataset:** CDC BRFSS 2015 · 253,680 U.S. adults  
**Goal:** Understand the data structure, class distribution, feature characteristics, and identify the core challenge of this dataset — the prediabetes detection problem.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded OK')

## 1. Load & Overview

In [ ]:
df = pd.read_csv('../data/diabetes_012_health_indicators_BRFSS2015.csv')
df['Diabetes_012'] = df['Diabetes_012'].astype(int)

print(f'Shape: {df.shape}')
print(f'No missing values: {df.isnull().sum().sum() == 0}')
print(f'\nDtypes: all float64 → {(df.dtypes == "float64").sum()} cols (cast to int where needed)')
df.describe().round(2)

## 2. Target Distribution — The Core Challenge

This is the most important EDA finding: **prediabetes represents only 1.8% of the dataset (n=4,631)**. This is not just a statistical imbalance — it reflects the true population prevalence of self-reported prediabetes, and it carries a direct clinical implication: any model that fails to account for this will effectively ignore the prediabetes class entirely.

In [ ]:
counts = df['Diabetes_012'].value_counts().sort_index()
labels = ['No diabetes', 'Prediabetes', 'Diabetes']
pcts = counts / len(df) * 100

print('Target distribution:')
for i, (label, n, p) in enumerate(zip(labels, counts.values, pcts.values)):
    print(f'  Class {i} — {label:<15} {n:>7,}  ({p:.1f}%)')

print(f'\nImbalance ratio (No DM / Prediabetes): {counts[0]/counts[1]:.0f}:1')
print(f'→ This is severe. SMOTE and balanced metrics required.')

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4C72B0', '#FF8C00', '#C44E52']
bars = ax.bar(labels, counts.values, color=colors, alpha=0.85, edgecolor='white')
for bar, val, pct in zip(bars, counts.values, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
            f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
ax.set_title('Target Class Distribution — BRFSS 2015', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 3. Feature Distributions by Class

Key question: do any features show meaningful separation across the three classes? This informs which variables will drive model predictions and whether prediabetes is separable from the other classes using these variables.

In [ ]:
continuous_feats = ['BMI', 'GenHlth', 'Age', 'MentHlth', 'PhysHlth', 'Income', 'Education']
class_colors = ['#4C72B0', '#FF8C00', '#C44E52']
class_labels = ['No diabetes', 'Prediabetes', 'Diabetes']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(continuous_feats):
    for cls, color, label in zip([0,1,2], class_colors, class_labels):
        subset = df[df['Diabetes_012'] == cls][feat]
        axes[i].hist(subset, bins=20, alpha=0.5, color=color, label=label, density=True)
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_ylabel('Density')
    if i == 0:
        axes[i].legend(fontsize=7)

axes[-1].set_visible(False)
fig.suptitle('Feature Distributions by Diabetes Status', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Key observation:')
print('  Prediabetes (orange) overlaps almost completely with No diabetes (blue) across all features.')
print('  This visual confirms the detection challenge even before modeling.')

## 4. Binary Feature Prevalence by Class

For binary features, compare the positive rate across the three diabetes classes.

In [ ]:
binary_feats = ['HighBP','HighChol','Smoker','Stroke','HeartDiseaseorAttack',
                'PhysActivity','Fruits','Veggies','HvyAlcoholConsump',
                'AnyHealthcare','NoDocbcCost','DiffWalk']

prevalence = pd.DataFrame({
    label: df[df['Diabetes_012']==cls][binary_feats].mean() * 100
    for cls, label in zip([0,1,2], class_labels)
})

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(binary_feats))
width = 0.28
for i, (label, color) in enumerate(zip(class_labels, class_colors)):
    ax.bar(x + i*width, prevalence[label], width, label=label, color=color, alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(binary_feats, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Prevalence (%)')
ax.set_title('Binary Feature Prevalence by Diabetes Status', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('Top risk markers (highest % difference between No DM and Diabetes):')
diff = (prevalence['Diabetes'] - prevalence['No diabetes']).sort_values(ascending=False)
for feat, val in diff.head(5).items():
    print(f'  {feat:<25} +{val:.1f}pp higher in diabetes')

## 5. Correlation Matrix

In [ ]:
corr = df.corr()
outcome_corr = corr['Diabetes_012'].drop('Diabetes_012').sort_values(ascending=False)

print('Feature correlation with Diabetes_012:')
for feat, val in outcome_corr.items():
    bar = '█' * int(abs(val) * 30)
    print(f'  {feat:<28} {val:+.3f}  {bar}')

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.3, annot_kws={'size': 7})
ax.set_title('Pearson Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. The Prediabetes Detection Problem — Statistical Evidence

Kruskal-Wallis test comparing distributions across classes, with post-hoc focus on No DM vs Prediabetes separation.

In [ ]:
from scipy.stats import kruskal, mannwhitneyu

print('Mann-Whitney U: No Diabetes vs Prediabetes — can features separate these classes?')
print(f'{"Feature":<28} {"Mean NoDM":>10} {"Mean PreDM":>12} {"p-value":>10} {"Effect":>8}')
print('-' * 72)

no_dm = df[df['Diabetes_012'] == 0]
pre_dm = df[df['Diabetes_012'] == 1]

for feat in continuous_feats:
    g0 = no_dm[feat]
    g1 = pre_dm[feat]
    stat, p = mannwhitneyu(g0, g1, alternative='two-sided')
    # rank-biserial correlation as effect size
    n1, n2 = len(g0), len(g1)
    r = 1 - (2 * stat) / (n1 * n2)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'{feat:<28} {g0.mean():>10.2f} {g1.mean():>12.2f} {p:>10.4f} {r:>7.3f} {sig}')

print('\nConclusion: Even statistically significant differences show near-zero effect sizes.')
print('Prediabetes and No Diabetes are statistically distinguishable but practically inseparable.')
print('This is the fundamental constraint of survey-only data for prediabetes detection.')

## 7. Equity Variables — Income, Education, Demographics

BRFSS includes socioeconomic variables. These matter clinically and will be used in the equity analysis in notebook 03.

In [ ]:
equity_vars = ['Income', 'Education', 'Sex', 'Age']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, var in zip(axes, equity_vars):
    dm_rate = df.groupby(var)['Diabetes_012'].apply(lambda x: (x == 2).mean() * 100)
    ax.bar(dm_rate.index, dm_rate.values, color='#C44E52', alpha=0.8)
    ax.set_title(f'Diabetes % by {var}', fontweight='bold')
    ax.set_xlabel(var)
    ax.set_ylabel('Diabetes prevalence (%)')

plt.suptitle('Diabetes Prevalence by Socioeconomic & Demographic Variables', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Clinical and policy relevance:')
print('  Income: inverse relationship — lower income → higher diabetes prevalence')
print('  Education: inverse relationship — consistent with social determinants of health literature')
print('  Age: strong positive relationship — highest prevalence in 70+ age groups')
print('  Sex: males slightly higher prevalence in some age bands')

## 8. EDA Summary

| Finding | Implication |
|---|---|
| Prediabetes = 1.8% of sample | SMOTE required; balanced accuracy as primary metric |
| Prediabetes barely separable from No DM | Expected low recall for class 1; document as clinical finding |
| GenHlth, BMI, Age strongest correlates | Will dominate feature importance |
| HighBP, HighChol high delta | Strong risk markers for diabetes class |
| Income/Education gradient | Equity analysis warranted |
| No missing values | No imputation needed |

**→ Proceed to notebook 02: Preprocessing & SMOTE**